<a id="unit-u.85f418ef962c4f838d6dd5b45414592a"></a>

<div>APACHE DORIS КӨМЕГІМЕН ДЕРЕКТЕР ҚОЙМАСЫН ҚҰРУ · ҚОСЫМША ЗЕРТХАНАЛЫҚ ЖҰМЫС</div>

<a id="optional-lab-5a-kafka--routine-load--doris"></a>

# Қосымша 5A жұмысы: Kafka → Routine Load → Doris

**5-модуль: деректерді пакеттік және үздіксіз жүктеу** · [Сабақ](course.md) · [Тест](quiz05_load_methods_and_retry_safety.ipynb) · [Оқу: үздіксіз жүктеу](optional05_continuous_ingestion.md)

Kafka topic-і тапсырыс оқиғаларын бір-бірден қабылдайды. Doris ішіндегі Routine Load тапсырмасы оларды үздіксіз тұтынып, шағын пакеттермен жазады және topic-тегі өз орнын сақтайды. Тапсырыстарды жіберіп, тапсырма барысын оқисыз, кезек жинау үшін кідіртесіз, жалғастырасыз, қайталану мен жаңартуды жібересіз және қате хабар тапсырмаға қалай әсер ететінін көресіз.

Шамамен 30 минут жоспарлаңыз. Жұмыс таңдау бойынша: басқа жұмыстар оған тәуелді емес.

**Не қажет**

- Кемінде 8 GiB жады және 4 CPU бар Docker (Docker Desktop: Settings → Resources). Бұл жұмыста Apple silicon бар Docker Desktop-та өлшенген ең жоғары пайдалану: Doris 2.9 GiB, Kafka 0.4 GiB. Бірінші ұяшық ештеңе іске қоспас бұрын Docker ресурстарын тексереді.
- Алғаш жүктеуге шамамен 640 MB (`apache/kafka:3.9.0`).
- 1-зертханалық жұмыстың курс сынақ ортасы. Бірінші ұяшық Doris контейнерін 6 GiB жады шегімен қайта жасайды; деректері сақталады. Алдымен басқа ноутбуктерді аяқтаңыз. 1-зертханалық жұмыстың **Іске қосу және қосылу** ұяшығы бұл шекті қайта алып тастайды: негізгі жұмыстарға оралмас бұрын оны орындаңыз, себебі олардың BE-сіне шек қалдыратын 2.4 GiB-тан көбірек қажет.

Жұмыс тек `ext_kafka_orders`, ал жаттығуда `ext_kafka_practice` кестесін қайта жасайды; екеуі де `dw_course_l1_streaming` дерекқорында, сондықтан негізгі курс кестелері өзгермейді. Әр орындау жаңа topic пен жаңа тапсырма қолданады. Қызметтері — `doris-warehousing-streaming` Compose жобасы: жұмыстың соңында тоқтату тәсілі, курстың README файлындағы [Тазалау](../../README.md#clean-up) бөлімінде жою тәсілі берілген.

Жұмысты орындай алмасаңыз да оқыңыз: әр қадамның **Күтілетін нәтижесі** орындалу кезінде не шығатынын көрсетеді.


In [ ]:
from pathlib import Path
import os, sys
from uuid import uuid4
course = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "dw_course").is_dir())
sys.path.insert(0, str(course))
from dw_course.docker_runtime import prepare_environment, connect_sandbox
from dw_course.runtime import expect
from dw_course.streaming import (
    prepare_streaming, kafka, produce, wait_rows, check_routine_load, cleanup_kafka_routine_load,
)
prepare_environment(start=True, streaming=True, streaming_profile="kafka")
os.environ["DW_DATABASE"] = "dw_course_l1_streaming"
lab = connect_sandbox()
print("Isolated lab database:", lab.database)

prepare_streaming("kafka", start=True)

<a id="unit-u.67a918b63f1a44269660c606b99186bc"></a>

**Күтілетін нәтиже**

`Docker capacity: … GiB RAM, … CPUs`, кемінде 8 GiB және 4 CPU, содан кейін алты жасыл қадамы бар іске қосу панелі. Алғаш орындағанда Compose Doris контейнерін 6 GiB шегімен қайта жасайды, оған шамамен 20 секунд кетеді. Содан кейін `Isolated lab database: dw_course_l1_streaming`, ал Compose алғаш орындағанда Kafka бейнесін жүктеп, `doris-warehousing-streaming-kafka-1` іске қосады.

Жады аз болса, ұяшық ештеңе іске қоспай тұрып тоқтап, жұмысқа қанша қажет екенін айтады.


<a id="unit-u.495d02f09c2342479e132b5094a47a21"></a>

<a id="1-create-the-target-table-a-topic-and-the-load-job"></a>

## 1. Мақсатты кестені, topic-ті және жүктеу тапсырмасын жасау

Баптау алдымен атаулары мен мақсатты кестелерін тексеріп, осы жұмыстың бұрынғы орындауы қалдырған ескі тапсырмаларды тоқтатады; басқа Routine Load тапсырмасына тимейді. Содан кейін әр тапсырысқа бір жол сақтайтын `ext_kafka_orders` Unique Key кестесін қайта жасайды, бір бөлімді topic жасап, тапсырманы жасайды:

| Тапсырма параметрі | Мағынасы |
| --- | --- |
| `"format"="json"` | Әр Kafka хабары — бір JSON нысаны; кілттері — баған атаулары |
| `"max_batch_interval"="5"` | Әр жүктеу жұмысы оқығанын бекітпес бұрын ең көбі 5 секунд орындалады |
| `"max_error_number"="0"`, `"strict_mode"="true"` | Кестеге сәйкес келмейтін алғашқы жолда тапсырманы кідірту |
| `"property.kafka_default_offsets"="OFFSET_BEGINNING"` | Жаңа тапсырма әр бөлімнің басынан басталады |

Doris тапсырманың орнын, яғни **барысын**, әр бекітілген жұмыспен бірге өзі сақтайды; бұл қарапайым Kafka consumer group бекіткен offset емес.


In [ ]:
cleanup_kafka_routine_load(lab)
expect(lab.query("SHOW ROUTINE LOAD"), [], title="No earlier Routine Load job is still active")
lab.execute("DROP TABLE IF EXISTS ext_kafka_orders")
lab.execute("""CREATE TABLE ext_kafka_orders (
    order_id BIGINT NOT NULL, customer_id BIGINT NOT NULL,
    order_status VARCHAR(32) NOT NULL, order_amount DECIMAL(18,2) NOT NULL
) UNIQUE KEY(order_id) DISTRIBUTED BY HASH(order_id) BUCKETS 1
PROPERTIES("replication_num"="1", "enable_unique_key_merge_on_write"="true")""")
run_id = uuid4().hex[:12]
topic = "course_orders_" + run_id
job_name = "course_orders_" + run_id
print(kafka("kafka-topics.sh", "--create", "--topic", topic, "--partitions", "1", "--replication-factor", "1"))
load_sql = f'''CREATE ROUTINE LOAD {job_name} ON ext_kafka_orders
COLUMNS(order_id, customer_id, order_status, order_amount)
PROPERTIES("format"="json", "desired_concurrent_number"="1",
           "max_batch_interval"="5", "max_error_number"="0", "strict_mode"="true")
FROM KAFKA("kafka_broker_list"="course-stream-kafka:9092", "kafka_topic"="{topic}",
           "property.kafka_default_offsets"="OFFSET_BEGINNING")'''
print(load_sql)
lab.execute(load_sql);

<a id="unit-u.67ce6eff6cb54496b0d38010d6070da3"></a>

**Күтілетін нәтиже**

Бұрынғы Routine Load тапсырмалары белсенді емес екенін көрсететін жасыл белгі, содан кейін `Created topic course_orders_<12 hexadecimal digits>.`, мұнда topic атауларындағы нүкте мен астыңғы сызық туралы Kafka ескертуінің мәні жоқ, содан кейін `CREATE ROUTINE LOAD` операторы. Тапсырма мен topic атауы бірдей. Қате болмаса, тапсырма бар және хабарларды күтеді.


<a id="unit-u.7263bda145c04b7a9736702c92bba747"></a>

<a id="2-send-two-orders-and-wait-until-they-are-visible"></a>

## 2. Екі тапсырыс жіберіп, көрінгенше күту

Өндіруші Kafka контейнерінің ішінде жұмыс істеп, екі JSON хабарын жазады. Kafka-дағы хабар әлі Doris жолы емес: келесі ұяшық екі жолды да сұрауға болғанша, ең көбі 3 минут күтеді және әр әрекетте тапсырманы тексереді. Тапсырма кідірсе не тоқтаса, күту тапсырма себебі мен қате URL-ін көрсетіп тоқтайды.


In [ ]:
rows = [dict(order_id=910001, customer_id=1, order_status="CREATED", order_amount="100.00"),
        dict(order_id=910002, customer_id=2, order_status="CREATED", order_amount="200.00")]
produce(topic, rows)
baseline = [[910001,1,"CREATED","100.00"],[910002,2,"CREATED","200.00"]]
expect(wait_rows(lab,"ext_kafka_orders",baseline, check_health=lambda: check_routine_load(lab, job_name), description="initial Kafka rows"), baseline,
       title="Orders 910001 and 910002 are visible")
lab.sql(f"SHOW ROUTINE LOAD FOR {job_name}", title="Consumption state and progress",
        columns=["Name", "State", "Progress", "Lag", "Statistic", "ReasonOfStateChanged"]);

<a id="unit-u.72580563e4f54286b1294ebf9414443b"></a>

**Күтілетін нәтиже**

Күту карточкасы шамамен 10 секунд ішінде жасыл болады, жасыл белгі екі 910001 және 910002 жолын растайды. Тапсырма жолында `State` RUNNING және `Progress` `{"0":"1"}` бар: 0-бөлімде тапсырма 1-offset-ке, екінші хабарға дейін жүктеді, себебі offset-тер 0-ден саналады. `Lag` `{"0":0}` ешбір хабар күтіп тұрмағанын білдіреді, `Statistic` ішінде `"loadedRows":2` және `"errorRows":0` көрсетіледі.


<a id="unit-u.46be49bff4c64ded8fddecee9bf7b1a2"></a>

<a id="3-pause-the-job-and-build-a-backlog"></a>

## 3. Тапсырманы кідірту және кезек жинау

Тапсырманы кідіртіп, кідіргенін тексеріңіз, содан кейін үшінші тапсырысты жіберіңіз. 12 секундтан, яғни екі пакет аралығынан ұзақ уақыттан кейін кестеде әлі тек алғашқы екі жол болуы керек. Кідірту тапсырма мен барысын сақтайды және бұрын жүктелген жолдарды жоймайды.


In [ ]:
import time
lab.execute(f"PAUSE ROUTINE LOAD FOR {job_name}")
paused = lab.sql(f"SHOW ROUTINE LOAD FOR {job_name}", title="The paused job",
                 columns=["Name", "State", "PauseTime", "Progress", "Lag"])
expect(paused.iloc[0]["State"], "PAUSED", title="The job is paused")
produce(topic,[dict(order_id=910003,customer_id=3,order_status="CREATED",order_amount="50.00")])
time.sleep(12)
expect(lab.query("SELECT * FROM ext_kafka_orders ORDER BY order_id"), baseline,
       title="The third order waits in Kafka")

<a id="unit-u.a4c666f6fa4f41efa90d06fb7bbbd624"></a>

**Күтілетін нәтиже**

`State` PAUSED, `PauseTime` бар. Он екі секундтан кейін кестеде әлі 910001 және 910002 бар: үшінші хабар Kafka-да, тапсырма барысынан кейін күтеді.


<a id="unit-u.45ee2ffaf17748a785534dca4f60df42"></a>

<a id="4-resume-then-send-a-duplicate-and-an-update"></a>

## 4. Жалғастыру, содан кейін қайталану мен жаңартуды жіберу

Тапсырманы жалғастырыңыз: ол өз барысынан жалғасып, күтіп тұрған тапсырысты жүктейді. Содан кейін 910001 тапсырысын өзгеріссіз қайта жіберіңіз, тағы бір рет PAID күйімен жіберіңіз. Кесте әр тапсырысқа бір жол сақтайды, сондықтан қайталану ештеңені өзгертпейді, жаңарту тапсырыс жолын ауыстырады. Unique Key кестесі әр тапсырыстың тарихын емес, соңғы күйін сақтайды; 7-модуль ретсіз келген оқиғаларды нұсқалармен шешеді.


In [ ]:
lab.execute(f"RESUME ROUTINE LOAD FOR {job_name}")
expected = baseline + [[910003,3,"CREATED","50.00"]]
expect(wait_rows(lab,"ext_kafka_orders",expected, check_health=lambda: check_routine_load(lab, job_name), description="new row after resume"),expected)
produce(topic,[rows[0],dict(order_id=910001,customer_id=1,order_status="PAID",order_amount="100.00")])
expected[0][2] = "PAID"
expect(wait_rows(lab,"ext_kafka_orders",expected, check_health=lambda: check_routine_load(lab, job_name), description="updated order after resume"),expected)
lab.sql(f"SHOW ROUTINE LOAD FOR {job_name}", title="Progress after resuming",
        columns=["Name", "State", "Progress", "Lag", "Statistic", "ReasonOfStateChanged"]);
expect(lab.query("SELECT COUNT(*),SUM(order_amount) FROM ext_kafka_orders"), [[3,"350.00"]],
       title="Three orders, 350.00")

<a id="unit-u.1aa8c97c44b64b63b03fe37b511042bc"></a>

**Күтілетін нәтиже**

Екі жасыл күту: `RESUME` кейін 910003 келеді, содан кейін 910001 PAID болады. Тапсырма жолында `State` RUNNING, `Progress` `{"0":"4"}`, 0–4 offset-тері бар бес хабар, `Lag` `{"0":0}` және `Statistic` ішінде `"loadedRows":5` көрсетіледі. Кестеде жалпы сомасы 350.00 болатын 3 жол бар: бес хабар, бірақ қайталану мен жаңарту 910001 жолын ауыстырды, сондықтан жол саны қанша хабар келгенін көрсетпейді.


<a id="unit-u.ae39db53b40f44e38d023e0d788a846f"></a>

<a id="5-practice-a-message-that-the-table-rejects"></a>

## 5. Жаттығу: кесте қабылдамайтын хабар

Сол параметрлермен екінші `course_practice_<12 hexadecimal digits>` topic-і мен тапсырмасын жасаңыз, ол `ext_kafka_orders` бағандары бар жаңа `ext_kafka_practice` кестесіне жүктейді. Дұрыс 930001 тапсырысын және сомасы `not-a-number` болатын 930002 тапсырысын жіберіңіз.

Орындамас бұрын болжаңыз: тапсырма 930001-ді жүктей ме? Соңында қандай күйде болады? Содан кейін `State`, `ReasonOfStateChanged`, `Statistic` және `ErrorLogUrls` оқып, қате журналын ашыңыз, соңында осы жұмыстың келесі орындауы сіздің тапсырмаңызды таппауы үшін тапсырманы тоқтатып, topic-ті жойыңыз. Қате URL-і контейнер ішіндегі BE-нің `127.0.0.1:8040` мекенжайын атайды; ноутбуктен сол жолды `os.environ["DW_BE_HTTP_URL"]` арқылы оқыңыз.


In [ ]:
# Your code here

<a id="unit-u.5c3a8032e07740e6ae9b324acafd89a4"></a>

Келесі ұяшықта үлгі шешім бар; ол жиналған күйде. Салыстыру үшін ашыңыз немесе орындаңыз.


In [ ]:
import requests
from dw_course.streaming import routine_load_state, wait_for

practice_job = "course_practice_" + uuid4().hex[:12]
lab.execute("DROP TABLE IF EXISTS ext_kafka_practice")
lab.execute("""CREATE TABLE ext_kafka_practice (
    order_id BIGINT NOT NULL, customer_id BIGINT NOT NULL,
    order_status VARCHAR(32) NOT NULL, order_amount DECIMAL(18,2) NOT NULL
) UNIQUE KEY(order_id) DISTRIBUTED BY HASH(order_id) BUCKETS 1
PROPERTIES("replication_num"="1", "enable_unique_key_merge_on_write"="true")""")
print(kafka("kafka-topics.sh", "--create", "--topic", practice_job, "--partitions", "1", "--replication-factor", "1"))
lab.execute(f'''CREATE ROUTINE LOAD {practice_job} ON ext_kafka_practice
COLUMNS(order_id, customer_id, order_status, order_amount)
PROPERTIES("format"="json", "desired_concurrent_number"="1",
           "max_batch_interval"="5", "max_error_number"="0", "strict_mode"="true")
FROM KAFKA("kafka_broker_list"="course-stream-kafka:9092", "kafka_topic"="{practice_job}",
           "property.kafka_default_offsets"="OFFSET_BEGINNING")''')
produce(practice_job, [dict(order_id=930001, customer_id=1, order_status="CREATED", order_amount="10.00"),
                       dict(order_id=930002, customer_id=2, order_status="CREATED", order_amount="not-a-number")])
wait_for(lambda: routine_load_state(lab, practice_job)["State"], lambda state: state == "PAUSED",
         description="the practice job pauses at the bad message", timeout=120)
lab.sql(f"SHOW ROUTINE LOAD FOR {practice_job}", title="The practice job after the bad message",
        columns=["Name", "State", "Progress", "Statistic", "ReasonOfStateChanged", "ErrorLogUrls"]);
error_url = routine_load_state(lab, practice_job)["ErrorLogUrls"]
print(requests.get(os.environ["DW_BE_HTTP_URL"] + error_url[error_url.index("/api/"):], timeout=10).text)
lab.sql("SELECT * FROM ext_kafka_practice ORDER BY order_id", title="What the practice job loaded");
expect(lab.query("SELECT order_id FROM ext_kafka_practice"), [(930001,)],
       title="The valid order of the batch is loaded")
lab.execute(f"STOP ROUTINE LOAD FOR {practice_job}")
print(kafka("kafka-topics.sh", "--delete", "--topic", practice_job))

<a id="unit-u.995cabcd4db14f04b254249606b0605b"></a>

**Күтілетін нәтиже**

Шамамен 5 секунд ішінде тапсырма PAUSED болады. `ReasonOfStateChanged` мәні `current error rows is more than max_error_number or the max_filter_ratio is more than the value set`, `Statistic` ішінде `"loadedRows":1` және `"errorRows":1`, ал `Progress` `{"0":"1"}` болады. Қате журналы жолды атайды: `Reason: column(order_amount) value is incorrect while strict mode is true, src value is not-a-number. src line [930002 2 CREATED not-a-number]`.

`ext_kafka_practice` ішінде 930001 бар. `max_error_number` = 0 алғашқы қате жолда тапсырманы кідіртеді, бірақ сол жұмыста жүктелген дұрыс жолдарды кері қайтармайды, әрі барыс қате хабардан өтіп кеткен болады: `RESUME` одан кейін жалғасар еді, сондықтан түзетілген тапсырысты қайта жіберу керек. 5-зертханалық жұмыстағы Stream Load-пен салыстырыңыз: онда `strict_mode` және `max_filter_ratio` = 0 бүкіл пакетті қабылдамайды.


<a id="unit-u.8bb85f604324404a9e1623acb337c9a4"></a>

<a id="6-stop-the-job-and-keep-the-results"></a>

## 6. Тапсырманы тоқтату және нәтижелерді сақтау

`STOP` тапсырманы біржола аяқтайды: кідіртілгеннен өзгеше, тоқтатылған тапсырманы жалғастыру мүмкін емес, жаңа тапсырма өз әдепкі offset-терінен бастар еді. Ұяшық осы орындаудың тапсырмасын тоқтатып, тек осы орындаудың topic-ін жояды және қосылымды жабады; жолдар Doris-те қалады.


In [ ]:
lab.execute(f"STOP ROUTINE LOAD FOR {job_name}")
print(kafka("kafka-topics.sh", "--delete", "--topic", topic))
lab.close()

<a id="unit-u.188200e3a354423eb3d3fec3a0968b66"></a>

**Күтілетін нәтиже**

Қате жоқ. `SHOW ROUTINE LOAD` тапсырманы енді тізбейді; `SHOW ALL ROUTINE LOAD` оны әлі STOPPED ретінде көрсетеді.


<a id="unit-u.1256e900e9004b80ae6d15e5d5e1cbf7"></a>

<a id="you-are-done-when"></a>

## Жұмысты аяқтау шарттары

- тапсырма кідіргенде үшінші тапсырыс неге Kafka-да қалғанын және тапсырма орнын қайда сақтағанын түсіндіре аласыз;
- Routine Load тапсырмасының `State`, `Progress`, `Lag` және `Statistic` өрістерін оқи аласыз;
- `max_error_number` = 0 тапсырмаға, сол жұмыстың жолдарына және барысына не істейтінін айта аласыз.

Kafka-ны тоқтатып, деректерін сақтау үшін курс каталогында мынаны орындаңыз:

```bash
docker compose --project-name doris-warehousing-streaming --file environments/streaming/compose.yml --profile kafka stop
```

Негізгі жұмыстарға оралмас бұрын 1-зертханалық жұмыстың **Іске қосу және қосылу** ұяшығын орындаңыз: ол Doris контейнерін 6 GiB шектеуінсіз қайта жасап, деректерін сақтайды.

**Бірдеңе дұрыс болмаса:**

- Бірінші ұяшық іске қосудан бас тартты: Docker-ге көрсетілген жады мен CPU санын беріңіз.
- `MEM_LIMIT_EXCEEDED` не `MEM_ALLOC_FAILED`: 1-зертханалық жұмыстың іске қосу ұяшығы Doris-тің 6 GiB шегін алып тастаған болуы мүмкін. Басқа ноутбуктерді аяқтап, осы жұмыстың бірінші ұяшығын қайта орындаңыз, ол шекті қайта орнатады; сонда `docker inspect doris-warehousing-course-doris-1 --format '{{.HostConfig.Memory}}'` 6442450944 көрсетеді.
- Тапсырма кідіргендіктен күту тоқтады: карточкасы `ReasonOfStateChanged`, `ErrorLogUrls` және қалған мәлімет үшін орындалатын операторды көрсетеді. Doris Kafka-ға курс Docker желісіндегі `course-stream-kafka:9092` арқылы қатынайды.
- Жұмыс үзілді: бірінші ұяшықтан қайта орындаңыз. Ол осы жұмыстың бұрынғы орындауы қалдырған `course_orders_<12 hexadecimal digits>` және `course_practice_<12 hexadecimal digits>` тапсырмаларын тоқтатады, басқа Routine Load тапсырмасына тимейді.
- Kafka журналы: `docker compose --project-name doris-warehousing-streaming --file environments/streaming/compose.yml logs --tail 100 kafka`.
Іске қосу және қосылу сынақ ортасының бұрынғы деректер көлі, бақылау тақтасы және сақтық көшірме желілік қосылымдарын да қалпына келтіреді. Ол қызметтерді бұрын жасасаңыз, көл сұрауы мен сақталған Superset диаграммалары әлі жүктелетінін тексеріңіз; деректері мен рұқсаттары сақталады.


<a id="unit-u.bb009e01a6a046d1a55f50ee4f7434d0"></a>

<a id="next"></a>

## Келесі қадам

Әлі тапсырмасаңыз, [үздіксіз жүктеу тестін](quiz05_continuous_ingestion.ipynb) тапсырыңыз немесе [5-модуль сабағына](course.md) оралыңыз не [Қосымша 5B жұмысын: MySQL → Flink CDC → Doris](optional05_flink_mysql_cdc.ipynb) орындап көріңіз. 6-модуль [6-модуль сабағынан](../module06-data-quality/course.md) басталады.
